# Modelo DW-ViT para Detección y Estimación de Sistemas Fotovoltaicos
Pipeline reproducible: verificación de GPU, dependencias, dataset COCO JSON, arquitectura **DW-ViT** y ciclo de entrenamiento con Early Stopping.

## 1. Verificación de Hardware

In [ ]:
!nvidia-smi

## 2. Instalación de Dependencias

In [ ]:
!pip install -q roboflow pycocotools timm scikit-learn pandas matplotlib pillow

## 3. Repositorio del Modelo DW-ViT

In [ ]:
import os
if not os.path.exists('DW-ViT'):
    !git clone https://github.com/pzhren/DW-ViT.git
%cd DW-ViT
if os.path.exists('DW-ViT.py'):
    !mv DW-ViT.py dw_vit.py
    !sed -i 's/class DW-ViT/class DWViT/g' dw_vit.py
print('✓ Repositorio y clase listos.')

## 4. Descarga del Dataset (COCO JSON)
Para este caso usaremos un dataset elaborado por fuente propia de imagenes satelitales a un zoom de 19 en la plataforma SASPlanet con imagenes recortadas de 640x640

In [ ]:
import getpass
from roboflow import Roboflow

api_key = getpass.getpass('API Key: ')
rf = Roboflow(api_key=api_key)
project = rf.workspace('daves-workspace-cvhyt').project('satellite-pv')
version = project.version(2)
dataset = version.download('coco')
print(f'✓ Dataset descargado en: {dataset.location}')

## 5. Dataset Loader de PyTorch

In [ ]:
import json, os, torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class CocoPVDataset(Dataset):
    def __init__(self, img_dir, ann_file, transform=None):
        self.img_dir = img_dir
        with open(ann_file, 'r') as f:
            self.coco = json.load(f)
        self.images = {img['id']: img for img in self.coco['images']}
        self.img_ids = list(self.images.keys())
        self.annotations = {}
        for ann in self.coco['annotations']:
            img_id = ann['image_id']
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann)
        self.transform = transform

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.images[img_id]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        image = Image.open(img_path).convert('RGB')
        num_boxes = len(self.annotations.get(img_id, []))
        if self.transform:
            image = self.transform(image)
        return image, num_boxes

transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_root = dataset.location
train_dataset = CocoPVDataset(os.path.join(data_root, 'train'), os.path.join(data_root, 'train', '_annotations.coco.json'), transform=transform)
val_dataset = CocoPVDataset(os.path.join(data_root, 'valid'), os.path.join(data_root, 'valid', '_annotations.coco.json'), transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
print(f'✓ Train: {len(train_dataset)} | Valid: {len(val_dataset)}')

## 6. Inicialización del Modelo DW-ViT

In [ ]:
from dw_vit import DWViT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DWViT(
    img_size=640,
    patch_size=4,
    in_chans=3,
    num_classes=1,
    embed_dim=96,
    depths=[2, 2, 6, 2],
    num_heads=[4, 8, 16, 32],
    window_size=[7, 7]
).to(device)
print(f'✓ Modelo cargado en {device}')

## 7. Entrenamiento (1000 Épocas + Early Stopping)

In [ ]:
import csv, numpy as np, torch.nn as nn

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
criterion = nn.MSELoss()
max_epochs, patience = 1000, 50
best_val_mae = float('inf')
patience_counter = 0

csv_file = 'dwvit_training_metrics.csv'
with open(csv_file, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'train_loss', 'val_mae'])

for epoch in range(1, max_epochs + 1):
    model.train()
    total_loss = 0.0
    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    model.eval()
    mae_errors = []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.clamp(outputs, min=0).cpu().numpy().flatten()
            reals = targets.numpy().flatten()
            mae_errors.extend(np.abs(preds - reals))

    val_mae = float(np.mean(mae_errors))
    with open(csv_file, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([epoch, avg_train_loss, val_mae])

    if val_mae < best_val_mae:
        best_val_mae = val_mae
        patience_counter = 0
        torch.save(model.state_dict(), 'dwvit_best_model.pth')
        status = '⭐ Checkpoint Guardado'
    else:
        patience_counter += 1
        status = f'Paciencia: {patience_counter}/{patience}'

    print(f'Época [{epoch:04d}/{max_epochs:04d}] | Loss: {avg_train_loss:.4f} | Val MAE: {val_mae:.4f} | {status}')
    if patience_counter >= patience:
        print(f'\n⏹️ Early Stopping activado en época {epoch}.')
        break
print(f'✓ Mejor Val MAE: {best_val_mae:.4f}')